# Chapter 30: What We Can Now Prove


In [ ]:
import sys
from pathlib import Path

ROOT = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "content" / "books" / "retrieval" / "_index.md").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError("repository root not found")

NOTEBOOK_DIR = ROOT / "notebooks" / "books" / "retrieval"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))
if str(ROOT / "examples") not in sys.path:
    sys.path.insert(0, str(ROOT / "examples"))

import _support
ROOT = _support.repo_root()
print("repo root:", ROOT)


In [ ]:

# Small read-only retrieval path with traceable refs and stage statuses.
from document_identity import Collection, PUBLIC, REFERENCE_FIELDS, digest_bytes, version_key

fixture = {
    "fixture_id": "toy-e2e",
    "sources": [{"source_id": "toy:handbook", "authority": "fictional"}],
    "versions": [{
        "source_id": "toy:handbook", "document_id": "d", "version_id": "v1",
        "text": "R17 reset: hold the blue button for ten seconds.", "capture_locator": "hand:d",
        "captured_at": "2026-10-17", "published_at": None,
    }],
    "snapshots": [], "observations": [], "continuity": [],
}
fixture["snapshots"] = [{"snapshot_id": "main", "observed_at": "2026-10-17", "documents": [{
    "source_id": "toy:handbook", "document_id": "d", "current_version": "v1",
    "scope": PUBLIC, "lifecycle": "active", "freshness": "current", "locator": "hand:d",
}]}]
collection = Collection(fixture)
from evidence_pipeline import whole_document_units, candidate_union, select, public_projection
units = whole_document_units(collection, "main")
hits = [{"unit": u, "origins": [{"method": "lexical", "rank": 1, "score": 1.0,
        "view": "main", "provenance": "toy", "representation_id": u["representation_id"]}]}
        for u in units]
union = candidate_union({"lexical": hits})
selected = select(union, max_units=1, characters=1000)
projected = public_projection(selected, units, collection, "main")
chain = {
    "available": len(units),
    "retrieved": len(hits),
    "eligible": len(projected["selected"]),
    "selected": len(selected["selected"]),
    "admitted": "not measured in this toy path",
    "used": "not measured",
    "helpful": "not measured",
}
print(chain)
for c in projected["selected"]:
    print("traceable ref:", {k: c["unit"][k] for k in REFERENCE_FIELDS})
assert chain["available"] >= chain["retrieved"] >= chain["eligible"]


In [ ]:

# Evidence-chain limits: success upstream does not imply downstream.
limits = {
    "selected_not_admitted_possible": True,
    "admitted_not_used_possible": True,
    "citation_not_causal_use": True,
    "answer_match_not_helpfulness": True,
}
print(limits)
assert all(limits.values())


In [ ]:

# Optional Pi tool invocation: safely gated and skipped by default.
import os, shutil, subprocess, json
from pathlib import Path

run_pi = os.environ.get("RUN_PI", "").lower() in ("1", "true", "yes")
index_path = ROOT / ".artifacts" / "capstone" / "runtime" / "teaching" / "index.json"
node = shutil.which("node")
print("RUN_PI:", run_pi, "node:", bool(node), "index:", index_path.is_file())
if not run_pi:
    print("SKIPPED: default path does not call Pi. Set RUN_PI=1 with prerequisites to invoke.")
elif not node or not index_path.is_file():
    print("SKIPPED: prerequisites missing (node or built teaching index).")
else:
    # Invoke the existing checked adapter only; no model-directed Pi turn.
    proc = subprocess.run(
        [node, str(ROOT / "integrations" / "pi" / "invoke.mjs")],
        input=json.dumps({"tool": "retrieval_search", "arguments": {"collection": "teaching", "query": "reciprocal rank fusion", "strategy": "lexical"}}),
        capture_output=True, text=True, timeout=60, cwd=str(ROOT))
    print("returncode:", proc.returncode)
    print("stdout preview:", proc.stdout[:500])
    assert proc.returncode == 0
print("Note: the book did not run a model-directed Pi turn.")


## Observed versus not established
Observed: a read-only evidence chain with traceable references and explicit unmeasured downstream stages. Not established: answer correctness, causal use, helpfulness, or any Pi model turn.